<a href="https://colab.research.google.com/github/kostismatz/GKS_ML_Project/blob/main/2_c_1BiRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.is_available())
import torch
print(torch.__version__)



True
2.10.0+cu128


In [2]:
# -*- coding: utf-8 -*-
"""

A RNN classifier applied to AG_NEWS dataset

Download dataset:
https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset

"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torch.utils.data import DataLoader
from torch import nn
from torch.nn import functional as F
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

MAX_WORDS = 25
EPOCHS = 15
LEARNING_RATE = 1e-3
BATCH_SIZE = 1024
EMBEDDING_DIM = 100
HIDDEN_DIM = 64

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

cuda


In [3]:
######################################################################
# Data processing
# -----------------------------


def tokenizer(text):
    return text.lower().split()

# All texts are truncated and padded to MAX_WORDS tokens
def collate_batch(batch):
    Y, X = zip(*batch)

    Y = torch.tensor(Y, dtype=torch.long) - 1

    X_processed = []

    for text in X:
        tokens = tokenizer(text)

        indices = []
        for token in tokens:
            if token in vocab:
                indices.append(vocab[token])
            else:
                indices.append(vocab["<UNK>"])

        if len(indices) < MAX_WORDS:
            indices += [vocab["<PAD>"]] * (MAX_WORDS - len(indices))
        else:
            indices = indices[:MAX_WORDS]

        X_processed.append(indices)

    X_tensor = torch.tensor(X_processed, dtype=torch.long)

    return X_tensor.to(device), Y.to(device)

train_dataset = [(label,train_data['Title'][i] + ' ' + train_data['Description'][i]) for i,label in enumerate(train_data['Class Index'])]
test_dataset = [(label,test_data['Title'][i] + ' ' + test_data['Description'][i]) for i,label in enumerate(test_data['Class Index'])]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, collate_fn=collate_batch)

target_classes = ["World", "Sports", "Business", "Sci/Tech"]

def build_vocabulary(dataset, min_freq=10):
    counter = Counter()

    for _, text in dataset:
        tokens = tokenizer(text)
        counter.update(tokens)

    vocab_dict = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    idx = 2
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab_dict[token] = idx
            idx += 1

    return vocab_dict

vocab = build_vocabulary(train_dataset, min_freq=10)

######################################################################

In [4]:
class model(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, output_dim):
        super(model, self).__init__()

        self.hidden_dim = hidden_dim

        self.embedding_layer = nn.Embedding(input_dim, embedding_dim)

        # 🔥 BiRNN
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # 🔥 2x hidden λόγω forward + backward
        self.linear = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, X_batch):
        embeddings = self.embedding_layer(X_batch)

        output, hidden = self.rnn(embeddings)

        # 🔥 καλύτερο pooling BiRNN
        forward = output[:, -1, :self.hidden_dim]
        backward = output[:, 0, self.hidden_dim:]

        out = torch.mean(output, dim=1)
        logits = self.linear(out)

        return logits

In [5]:

     # Initiate an instance of the model
# ---------------------------------


classifier = model(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, len(target_classes)).to(device)
# Define loss function and opimization algorithm
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([param for param in classifier.parameters() if param.requires_grad == True],lr=LEARNING_RATE)

# Count model parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('\nModel:')
print(classifier)
print('Total parameters: ',count_parameters(classifier))
print('\n\n')

######################################################################


Model:
model(
  (embedding_layer): Embedding(25099, 100)
  (rnn): RNN(100, 64, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=128, out_features=4, bias=True)
)
Total parameters:  2531664





In [6]:
# Define functions to train and evaluate the model
# ------------------------------------------------


def EvaluateModel(model, loss_fn, val_loader):
    model.eval()
    with torch.no_grad():

        Y_actual, Y_preds, losses = [], [], []

        for X, Y in val_loader:

            # 🔥 MOVE TO GPU
            X = X.to(device)
            Y = Y.to(device)

            preds = model(X)
            loss = loss_fn(preds, Y)

            losses.append(loss.item())

            Y_actual.append(Y)
            Y_preds.append(preds.argmax(dim=-1))

        Y_actual = torch.cat(Y_actual)
        Y_preds = torch.cat(Y_preds)

    return torch.tensor(losses).mean(), Y_actual.cpu().numpy(), Y_preds.cpu().numpy()



def TrainModel(model, loss_fn, optimizer, train_loader, epochs):
    epoch_times = []

    for i in range(1, epochs+1):
        model.train()
        print('Epoch:', i)

        losses = []

        start_time = time.time()

        for X, Y in tqdm(train_loader):

            # 🔥 MOVE TO GPU
            X = X.to(device)
            Y = Y.to(device)

            Y_preds = model(X)
            loss = loss_fn(Y_preds, Y)

            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        end_time = time.time()

        epoch_time = end_time - start_time
        epoch_times.append(epoch_time)

        print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
        print("Epoch Time (sec): {:.2f}".format(epoch_time))

    print("\nAverage Epoch Time:", sum(epoch_times)/len(epoch_times))

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

TrainModel(classifier, loss_fn, optimizer, train_loader, EPOCHS)

######################################################################

Epoch: 1


100%|██████████| 118/118 [00:02<00:00, 45.99it/s]


Train Loss : 0.983
Epoch Time (sec): 2.57
Epoch: 2


100%|██████████| 118/118 [00:02<00:00, 50.51it/s]


Train Loss : 0.535
Epoch Time (sec): 2.34
Epoch: 3


...

Train Loss : 0.127
Epoch Time (sec): 3.08
Epoch: 14


100%|██████████| 118/118 [00:03<00:00, 38.66it/s]


Train Loss : 0.116
Epoch Time (sec): 3.06
Epoch: 15


100%|██████████| 118/118 [00:02<00:00, 50.37it/s]

Train Loss : 0.103
Epoch Time (sec): 2.35

Average Epoch Time: 2.6187005996704102


In [7]:
# Evaluate the model with test dataset
# ------------------------------------


_, Y_actual, Y_preds = EvaluateModel(classifier, loss_fn, test_loader)

print("\nTest Accuracy : {:.3f}".format(accuracy_score(Y_actual, Y_preds)))
print("\nClassification Report : ")
print(classification_report(Y_actual, Y_preds, target_names=target_classes))
print("\nConfusion Matrix : ")
print(confusion_matrix(Y_actual, Y_preds))


Test Accuracy : 0.890

Classification Report : 
              precision    recall  f1-score   support

       World       0.90      0.89      0.90      1900
      Sports       0.94      0.95      0.94      1900
    Business       0.87      0.84      0.85      1900
    Sci/Tech       0.85      0.88      0.86      1900

    accuracy                           0.89      7600
   macro avg       0.89      0.89      0.89      7600
weighted avg       0.89      0.89      0.89      7600


Confusion Matrix : 
[[1700   54   78   68]
 [  38 1801   30   31]
 [  83   25 1597  195]
 [  66   37  133 1664]]


In [8]:
# =========================
# 🔁 3 RUN EXPERIMENT
# =========================

from sklearn.metrics import accuracy_score
import numpy as np

accuracies = []

for run in range(3):
    print(f"\n===== RUN {run+1} =====")

    # νέο model κάθε φορά
    classifier = model(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, len(target_classes)).to(device)

    optimizer = torch.optim.Adam(classifier.parameters(), lr=LEARNING_RATE)

    # training
    TrainModel(classifier, loss_fn, optimizer, train_loader, EPOCHS)

    # evaluation
    _, y_true, y_pred = EvaluateModel(classifier, loss_fn, test_loader)

    acc = accuracy_score(y_true, y_pred)
    accuracies.append(acc)

    print("Test Accuracy:", acc)


    print("\n===== FINAL RESULTS =====")

mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

print("Accuracies:", accuracies)
print("Mean Accuracy:", mean_acc)
print("Std:", std_acc)


===== RUN 1 =====
Epoch: 1


100%|██████████| 118/118 [00:03<00:00, 36.81it/s]


Train Loss : 0.972
Epoch Time (sec): 3.21
Epoch: 2


100%|██████████| 118/118 [00:02<00:00, 50.92it/s]


Train Loss : 0.537
Epoch Time (sec): 2.32
Epoch: 3


...

Train Loss : 0.124
Epoch Time (sec): 2.21
Epoch: 14


100%|██████████| 118/118 [00:02<00:00, 51.96it/s]


Train Loss : 0.110
Epoch Time (sec): 2.28
Epoch: 15


100%|██████████| 118/118 [00:03<00:00, 37.22it/s]


Train Loss : 0.099
Epoch Time (sec): 3.17

Average Epoch Time: 2.4316240310668946
Test Accuracy: 0.8846052631578948

===== FINAL RESULTS =====
Accuracies: [0.8889473684210526, 0.8892105263157895, 0.8846052631578948]
Mean Accuracy: 0.8875877192982456
Std: 0.0021116496696860568
